# Fooocus-sud в Colab

Движок (Fooocus + GPU) работает здесь, интерфейс открываешь в браузере ноута: **https://colab.period.pp.ua**
(обратный SSH-туннель Colab → VPS → Traefik).

**Секреты Colab** (значок 🔑 слева, включи «доступ из блокнота»):

| Имя | Что |
|---|---|
| `HF_TOKEN` | токен HuggingFace (для приватных моделей; без него качаются только публичные) |
| `VPS_SSH_KEY` | приватный ключ туннеля: весь текст файла `~/.ssh/colab_vps` с ноута |
| `FOOOCUS_USER` / `FOOOCUS_PASS` | *(необязательно)* второй пароль уже в самом Fooocus; Traefik и так спрашивает свой |

Запускай ячейки по порядку 1 → 4. Картинки сохраняются только в `/content` и пропадают вместе с сессией.


In [ ]:
# @title 1️⃣ Настройки + GPU + клон форка
BRANCH     = "main"  # @param {type:"string"}
DRIVE_BASE = "/content/drive/MyDrive/COLAB_MOUNTS/models/loras2"  # @param {type:"string"}
VPS_HOST   = "128.140.12.50"  # @param {type:"string"}
VPS_USER   = "root"  # @param {type:"string"}
VPS_BIND   = "172.18.0.1:8080"  # @param {type:"string"}
PUBLIC_URL = "https://colab.period.pp.ua"  # @param {type:"string"}

import os, subprocess

REPO        = "https://github.com/cerbcrowler/Fooocus-sud.git"
FOOOCUS_DIR = "/content/Fooocus"
PORT        = 7865

def sh(cmd, check=True):
    """Выполнить команду и показать вывод."""
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"Команда упала ({r.returncode}): {cmd}")
    return r

print("=== GPU ===")
if sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False).returncode != 0:
    print("❌ GPU нет! Среда выполнения → Сменить тип → GPU")

print(f"\n=== ФОРК: {REPO} ({BRANCH}) ===")
if os.path.isdir(f"{FOOOCUS_DIR}/.git"):
    sh(f"git -C {FOOOCUS_DIR} fetch origin {BRANCH}")
    sh(f"git -C {FOOOCUS_DIR} checkout -B {BRANCH} origin/{BRANCH}")
    sh(f"git -C {FOOOCUS_DIR} reset --hard origin/{BRANCH}")
else:
    sh(f"git clone -b {BRANCH} {REPO} {FOOOCUS_DIR}")
sh(f"git -C {FOOOCUS_DIR} log --oneline -1")
print("✅ Код на месте")


In [ ]:
# @title 2️⃣ Google Drive + модели (links.txt)
import sys, importlib
from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
print("✅ Drive смонтирован")

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print(f"✅ HF-токен: {HF_TOKEN[:7]}...")
except Exception as e:
    HF_TOKEN = None
    print(f"⚠️ HF-токена нет ({e}): приватные модели не скачаются")

if FOOOCUS_DIR not in sys.path:
    sys.path.insert(0, FOOOCUS_DIR)
import sud.models
importlib.reload(sud.models)  # подхватить свежий код после git pull
print()
sud.models.sync(FOOOCUS_DIR, DRIVE_BASE, hf_token=HF_TOKEN)


In [ ]:
# @title 3️⃣ Пароль + туннель на VPS
import json, time

# --- второй пароль, уже в самом Fooocus (необязательно: Traefik и так спрашивает пароль) ---
AUTH_JSON = f"{FOOOCUS_DIR}/auth.json"
try:
    auth = [{"user": userdata.get("FOOOCUS_USER"), "pass": userdata.get("FOOOCUS_PASS")}]
    with open(AUTH_JSON, "w") as f:
        json.dump(auth, f)
    print(f"✅ auth.json: пользователь {auth[0]['user']} (вход будет двойной: Traefik + Fooocus)")
except Exception:
    if os.path.exists(AUTH_JSON):
        os.remove(AUTH_JSON)
    print("ℹ️ Секретов FOOOCUS_USER/FOOOCUS_PASS нет: вход только по паролю Traefik")

# --- SSH-ключ (на VPS он ограничен: только туннель на VPS_BIND и статус, без шелла) ---
KEY = "/root/.ssh/id_vps"
os.makedirs("/root/.ssh", exist_ok=True)
import re

def normalize_key(raw):
    """Чинит ключ после Colab Secrets: убирает \\r, восстанавливает переносы строк."""
    raw = raw.replace("\r", "").strip()
    m = re.search(r"-----BEGIN ([A-Z ]+)-----(.*?)-----END \1-----", raw, re.S)
    if not m:
        raise ValueError("в VPS_SSH_KEY нет строк -----BEGIN/END----- (скопирован не весь файл?)")
    kind, body = m.group(1), re.sub(r"\s+", "", m.group(2))
    lines = "\n".join(body[i:i + 70] for i in range(0, len(body), 70))
    return f"-----BEGIN {kind}-----\n{lines}\n-----END {kind}-----\n"

with open(KEY, "w") as f:
    f.write(normalize_key(userdata.get("VPS_SSH_KEY")))
os.chmod(KEY, 0o600)
if sh(f"ssh-keygen -l -f {KEY}", check=False).returncode != 0:
    raise RuntimeError("❌ Ключ не читается: перескопируй ~/.ssh/colab_vps в секрет VPS_SSH_KEY")
SSH_OPTS = (f"-i {KEY} -o IdentitiesOnly=yes -o StrictHostKeyChecking=accept-new -o ServerAliveInterval=15 "
            f"-o ServerAliveCountMax=3 -o ConnectTimeout=10")
print("✅ Ключ для VPS записан")

# --- туннель с автопереподключением ---
loop = f"""#!/bin/bash
while true; do
  echo "$(date '+%F %T') подключаюсь к {VPS_HOST}"
  ssh -N {SSH_OPTS} -o ExitOnForwardFailure=yes \\
      -R {VPS_BIND}:localhost:{PORT} {VPS_USER}@{VPS_HOST}
  rc=$?
  echo "$(date '+%F %T') туннель упал (код $rc), повтор через 5 с"
  sleep 5
done
"""
with open("/content/tunnel_loop.sh", "w") as f:
    f.write(loop)

sh("pkill -f tunnel_loop.sh", check=False)
sh(f"pkill -f 'ssh -N.*{VPS_BIND}'", check=False)
subprocess.Popen("nohup bash /content/tunnel_loop.sh > /content/tunnel.log 2>&1 &", shell=True)
time.sleep(6)

print("\n=== ПРОВЕРКИ (статус с VPS) ===")
# ключ на VPS вместо шелла запускает /root/colab-status.sh: порт туннеля + статус Traefik
status = sh(f"ssh -T {SSH_OPTS} {VPS_USER}@{VPS_HOST}", check=False).stdout
print("✅ Туннель поднят" if "OK:" in status else "❌ Туннель НЕ поднят, смотри лог ниже")
print("\n--- tunnel.log ---")
_ = sh("tail -n 5 /content/tunnel.log", check=False)


In [ ]:
# @title 4️⃣ Запуск Fooocus
print(f"🌐 Интерфейс: {PUBLIC_URL}  (появится, когда ниже напишет 'App started')")
%cd /content/Fooocus
!PYTORCH_CUDA_ALLOC_CONF=garbage_collection_threshold:0.9,max_split_size_mb:512 python launch.py --port {PORT} --listen --always-high-vram --preset sud --disable-analytics
